# HuggingFace smolagents Practice
This notebook demonstrates how to use the `smolagents` framework to create a basic agent, define custom tools, integrate MCP tools, and observe the agent's behavior.

We will use an **Azure OpenAI** deployment (`gpt-5.4-mini` by default) for this tutorial.

## 1. Install and Import Dependencies
First, we install `smolagents` with the `openai` extra (client for OpenAI-compatible APIs such as Azure OpenAI) and the `toolkit` extra (default web search and webpage tools). We also install `python-dotenv` to load credentials from a `.env` file.

In [1]:
%pip install -q "smolagents[openai,toolkit]>=1.26" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from smolagents import CodeAgent, OpenAIModel, tool

C:\Users\artem.lubkivskyi\Documents\GitHub\Accenture\Day 14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure the Azure OpenAI Model
Azure OpenAI's v1 API (`https://<resource>.openai.azure.com/openai/v1/`) is OpenAI-compatible, so we use smolagents' `OpenAIModel` and point its `api_base` at the Azure endpoint. The `model_id` is your Azure **deployment name** (e.g. `gpt-5.4-mini` or `gpt-5.6-luna`).

**Note:** Store your credentials in a `.env` file (in this folder or any parent folder). If a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [3]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")

model = OpenAIModel(
    model_id=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 3. Create a Basic CodeAgent
We'll start by creating a simple `CodeAgent` without any custom tools. This agent can write and execute Python code to solve problems.

In [4]:
agent = CodeAgent(tools=[], model=model)
agent.run("Calculate the 10th Fibonacci number.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Calculate the 10th Fibonacci number.                                                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
The 10th Fibonacci number is **55**.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

[Step 1: Duration 1.66 seconds| Input tokens: 1,976 | Output tokens: 14]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  a, b = 0, 1                                                                                                      
  for _ in range(10):                                                                                              
      a, b = b, a + b                                                                                              
  final_answer(a)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 55

[Step 2: Duration 1.33 seconds| Input tokens: 4,089 | Output tokens: 83]

55

## 4. Define a Custom Tool
We can extend the agent's capabilities by defining custom tools. Here, we create a simple calculator tool using the `@tool` decorator.

In [5]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 15.0) -> str:
    """
    Calculates the tip amount and total bill.

    Args:
        bill_amount: The total cost of the meal.
        tip_percentage: The percentage of tip to leave (default is 15%).
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    return f"Tip: ${tip:.2f}, Total: ${total:.2f}"

# Test the tool directly
print(calculate_tip(100, 20))

Tip: $20.00, Total: $120.00


## 5. Integrate an MCP Tool
The Model Context Protocol (MCP) allows agents to connect to external tools standardized across different systems.

In `smolagents`, you can load tools from an MCP server with `MCPClient`.
```python
# Example of loading tools from an MCP server (requires `pip install "smolagents[mcp]"` and a running MCP server)
# from smolagents import MCPClient
# with MCPClient({"url": "http://localhost:8000/mcp", "transport": "streamable-http"}, structured_output=True) as mcp_tools:
#     agent = CodeAgent(tools=mcp_tools, model=model)
```

For this practice, we will simulate an MCP tool that retrieves system status.

In [6]:
@tool
def get_system_status(system_id: str) -> str:
    """
    Retrieves the status of a specific system.

    Args:
        system_id: The ID of the system to check (e.g., 'server-01').
    """
    # Simulated MCP response
    statuses = {"server-01": "Online", "server-02": "Maintenance", "db-prod": "Active"}
    return f"System {system_id} status: {statuses.get(system_id, 'Unknown')}"

## 6. Enable Observability
Observability is crucial for understanding how your agent makes decisions. `smolagents` provides built-in logging.
We can also use `GradioUI` for an interactive trace, but here we will rely on the console output which shows the agent's "thought" process.

When running the agent, `smolagents` prints the steps:
1.  **Thought**: The agent's reasoning.
2.  **Code**: The code generated to call a tool.
3.  **Observation**: The result from the tool.

In [7]:
# No specific code needed to enable basic logging, it's on by default in the run output.
# However, we can inspect the agent's logs after execution.
print("Observability is enabled by default. Watch the output of the next cell.")

Observability is enabled by default. Watch the output of the next cell.


## 7. Run the Agent with Tools
Now we combine everything. We create a new `CodeAgent` equipped with both our `calculate_tip` tool and the `get_system_status` tool.
We will give it a task that requires using both.

In [8]:
# Initialize agent with tools
final_agent = CodeAgent(
    tools=[calculate_tip, get_system_status],
    model=model,
    add_base_tools=True # Adds the default DuckDuckGo web search and webpage visit tools
)

# Run with a complex query
response = final_agent.run(
    "I had a meal costing $150. Calculate the tip at 20%. "
    "Also, check the status of 'server-01' to see if I can pay online."
)

print(f"Final Answer: {response}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I had a meal costing $150. Calculate the tip at 20%. Also, check the status of 'server-01' to see if I can pay  │
│ online.                                                                                                         │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            The tip at 20% on a $150 meal is **$30**, for a total of **$180**.

I can also check the status of **server-01** if you want, but I don’t have enough information yet from the current 
task context to verify online payment availability.</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 1: Duration 1.32 seconds| Input tokens: 2,213 | Output tokens: 69]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tip_info = calculate_tip(bill_amount=150, tip_percentage=20)                                                     
  status_info = get_system_status(system_id="server-01")                                                           
  final_answer(f"Tip result: {tip_info}\nServer status: {status_info}")                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Tip result: Tip: $30.00, Total: $180.00
Server status: System server-01 status: Online

[Step 2: Duration 1.10 seconds| Input tokens: 4,667 | Output tokens: 155]

Final Answer: Tip result: Tip: $30.00, Total: $180.00
Server status: System server-01 status: Online


## Conclusion
You have successfully created a `smolagents` agent powered by Azure OpenAI, defined custom tools, simulated an MCP tool integration, and observed the agent's execution flow.
Experiment with adding more tools or changing the model to see how the agent adapts!

## 8. Challenge: Build a Smart Home Controller
Now it's your turn! Apply what you've learned to build a simple smart home controller.

**Your Task:**
1.  Create a custom tool `get_room_temperature(room: str)` that returns a simulated temperature (e.g., "18°C").
2.  Create a custom tool `set_thermostat(room: str, target_temp: int)` that simulates setting the temperature and returns a confirmation message.
3.  Instantiate a `CodeAgent` with these tools and the Azure OpenAI model.
4.  Run the agent with a query: *"Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C."*

In [9]:
# Fake house: the thermostat really changes the room temperature,
# so I can check afterwards that the agent did what it said
house = {"living room": 18.5, "bedroom": 21.0, "kitchen": 19.5}


@tool
def get_room_temperature(room: str) -> str:
    """
    Returns the current temperature in a room of the house.

    Args:
        room: Name of the room, for example 'Living Room' or 'Bedroom'.
    """
    key = room.strip().lower()
    if key not in house:
        return f"Unknown room '{room}'. Rooms: {', '.join(house)}"
    return f"{house[key]}°C"


@tool
def set_thermostat(room: str, target_temp: int) -> str:
    """
    Sets the thermostat in a room to a target temperature.

    Args:
        room: Name of the room, for example 'Living Room'.
        target_temp: Wanted temperature in °C, between 10 and 30.
    """
    key = room.strip().lower()
    if key not in house:
        return f"Unknown room '{room}'."
    if not 10 <= target_temp <= 30:
        return f"{target_temp}°C is outside the allowed range (10-30°C)."
    old = house[key]
    house[key] = float(target_temp)
    return f"Thermostat in {room} set to {target_temp}°C (was {old}°C)."


print(get_room_temperature("Living Room"))
print(get_room_temperature("Garage"))

18.5°C
Unknown room 'Garage'. Rooms: living room, bedroom, kitchen


In [10]:
# Structured outputs: the model returns its thought and code as JSON fields instead of <code> tags,
# so a missing tag can't cause a parsing error
home_agent = CodeAgent(
    tools=[get_room_temperature, set_thermostat],
    model=model,
    use_structured_outputs_internally=True,
)

answer = home_agent.run(
    "Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C."
)
print("Answer:", answer)
print("House now:", house)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C.                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  temp_str = get_room_temperature('Living Room')                                                                   
  print(temp_str)                                                                                                  
                                                                                                                   
  # Try to extract the numeric temperature from the returned string.                                               
  import re                                                                                                        
  match = re.search(r'-?\d+(?:\.\d+)?', temp_str)                                                                  
  current_temp = float(match.group()) if match else None                                                           
                                                                                                                   
  if current_temp is not None and current_temp < 20:                                                               
      result = set_thermostat('Living Room', 22)                                                                   
      final_answer(result)                                                                                         
  else:                                                                                                            
      final_answer(f'No change needed. Living Room temperature is {temp_str}.')                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
18.5°C

Final answer: Thermostat in Living Room set to 22°C (was 18.5°C).

[Step 1: Duration 1.45 seconds| Input tokens: 1,964 | Output tokens: 193]

Answer: Thermostat in Living Room set to 22°C (was 18.5°C).
House now: {'living room': 22.0, 'bedroom': 21.0, 'kitchen': 19.5}


Same rule for a room that is already warm (bedroom is 21°C), here the agent should only read the temperature and not touch the thermostat.

In [11]:
answer = home_agent.run(
    "Check the temperature in the Bedroom. If it's below 20°C, set the thermostat to 22°C."
)
print("Answer:", answer)
print("House now:", house)
print("Steps in memory:", len(home_agent.memory.steps))

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Check the temperature in the Bedroom. If it's below 20°C, set the thermostat to 22°C.                           │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  temp_str = get_room_temperature('Bedroom')                                                                       
  print(temp_str)                                                                                                  
                                                                                                                   
  # Try to extract a numeric temperature from the returned string.                                                 
  import re                                                                                                        
  match = re.search(r'(-?\d+(?:\.\d+)?)', temp_str)                                                                
  current_temp = float(match.group(1)) if match else None                                                          
                                                                                                                   
  if current_temp is not None and current_temp < 20:                                                               
      result = set_thermostat('Bedroom', 22)                                                                       
      final_answer(result)                                                                                         
  else:                                                                                                            
      final_answer(f'Bedroom temperature is {temp_str}; no change needed.')                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
21.0°C

Final answer: Bedroom temperature is 21.0°C; no change needed.

[Step 1: Duration 1.73 seconds| Input tokens: 1,963 | Output tokens: 183]

Answer: Bedroom temperature is 21.0°C; no change needed.
House now: {'living room': 22.0, 'bedroom': 21.0, 'kitchen': 19.5}
Steps in memory: 2


### What I noticed

- `CodeAgent` writes Python instead of JSON tool calls. Both course agents above hit a parsing error in step 1 in this run: the model wrote only the answer (`The 10th Fibonacci number is **55**.</code>`) or answered the tip question in plain text, without the opening `<code>` tag. smolagents sent the error back and in step 2 the model wrote real code (55, and $30 tip / $180 total with server-01 Online).
- In my first run without the option, `home_agent` sent 4 empty code blocks before the real one and used about 11 100 input tokens for the living room. With `use_structured_outputs_internally=True` (thought and code come back as JSON fields, no tags) each task took one step and about 2 000 input tokens. It is one run each, so I can't say it never fails, but the difference is big.
- The living room code reads 18.5°C, compares with 20 and sets 22°C, and my `house` dict shows the change really happened. For the bedroom (21°C) it only read the temperature and answered "no change needed".
- `run()` starts with fresh memory each time (2 steps after the second run), so the agent does not remember the living room task.